In [ ]:
# ============================================================
# CUADERNO 07 — COMPARACION DE RESULTADOS
# Semana 10: Ejecutar Experimentos
# Proyecto: Modelos de ML para Detección de Anomalías
#           en el Tráfico y Logs de Entornos Cloud
# ============================================================

In [ ]:
# ── CELDA 1: Configuración ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
ruta_comparacion = f"{ruta_resultados}/comparacion"
os.makedirs(ruta_comparacion, exist_ok=True)

print("✅ Configuracion lista")

Mounted at /content/drive
✅ Configuracion lista


In [ ]:
# ── CELDA 2: Cargar resultados de los 4 modelos ──────────────
print("="*60)
print("CARGANDO RESULTADOS DE LOS 4 MODELOS")
print("="*60)

# Isolation Forest
df_if = pd.read_csv(
    f"{ruta_resultados}/IF_resultados_detallados.csv")

# Autoencoder
df_ae = pd.read_csv(
    f"{ruta_resultados}/AE_resultados_umbral_dinamico.csv")

# One-Class SVM
df_svm = pd.read_csv(
    f"{ruta_resultados}/OCSVM_resultados_detallados.csv")

# LSTM Autoencoder
df_lstm = pd.read_csv(
    f"{ruta_resultados}/LSTMAE_A100_resultados_detallados.csv")

print(f"✅ IF    : {len(df_if)} repeticiones")
print(f"✅ AE    : {len(df_ae)} repeticiones")
print(f"✅ SVM   : {len(df_svm)} repeticiones")
print(f"✅ LSTM  : {len(df_lstm)} repeticiones")

CARGANDO RESULTADOS DE LOS 4 MODELOS
✅ IF    : 5 repeticiones
✅ AE    : 5 repeticiones
✅ SVM   : 5 repeticiones
✅ LSTM  : 5 repeticiones


In [ ]:
# ── CELDA 3: Tabla comparativa principal ─────────────────────
print("="*60)
print("TABLA COMPARATIVA — 4 MODELOS")
print("="*60)

metricas_vi = [
    "Tiempo_entrenamiento_s",
    "Velocidad_inferencia_mps",
    "Uso_memoria_MB",
    "Uso_CPU_pct"
]

metricas_vd = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "FPR",
    "AUC_ROC"
]

modelos = {
    "Isolation Forest": df_if,
    "Autoencoder": df_ae,
    "One-Class SVM": df_svm,
    "LSTM Autoencoder": df_lstm
}

# Construir tabla
filas = []
for nombre, df in modelos.items():
    fila = {"Modelo": nombre}
    for m in metricas_vi + metricas_vd:
        if m in df.columns:
            media = df[m].mean()
            std = df[m].std()
            fila[m] = f"{media:.4f} ± {std:.4f}"
        else:
            fila[m] = "N/A"
    filas.append(fila)

df_comparacion = pd.DataFrame(filas)
df_comparacion = df_comparacion.set_index("Modelo")

# Mostrar VI
print("\n── VARIABLE INDEPENDIENTE (Costo Computacional) ──")
print(df_comparacion[metricas_vi].to_string())

# Mostrar VD
print("\n── VARIABLE DEPENDIENTE (Calidad de Detección) ──")
print(df_comparacion[metricas_vd].to_string())

TABLA COMPARATIVA — 4 MODELOS

── VARIABLE INDEPENDIENTE (Costo Computacional) ──
                 Tiempo_entrenamiento_s Velocidad_inferencia_mps         Uso_memoria_MB        Uso_CPU_pct
Modelo                                                                                                    
Isolation Forest       18.1061 ± 2.6679  212650.0500 ± 6970.0319      49.5940 ± 68.8346  15.0000 ± 30.8221
Autoencoder          385.1427 ± 83.8962    17085.6440 ± 101.0977    383.2760 ± 731.6258    5.1400 ± 4.5087
One-Class SVM          10.0299 ± 0.4034     11257.5140 ± 49.5568       1.9300 ± 28.2876    1.7200 ± 2.4077
LSTM Autoencoder   1083.5974 ± 193.0281    16341.3460 ± 149.0704  1350.2860 ± 2584.0937    2.2800 ± 3.8932

── VARIABLE DEPENDIENTE (Calidad de Detección) ──
                         Accuracy        Precision           Recall         F1_Score              FPR          AUC_ROC
Modelo                                                                                                    

In [ ]:
# ── CELDA 4: Tabla numérica para análisis ────────────────────
print("="*60)
print("TABLA NUMERICA — MEDIAS Y DESVIACIONES")
print("="*60)

filas_num = []
for nombre, df in modelos.items():
    fila = {"Modelo": nombre}
    for m in metricas_vi + metricas_vd:
        if m in df.columns:
            fila[f"{m}_media"] = round(df[m].mean(), 4)
            fila[f"{m}_std"] = round(df[m].std(), 4)
        else:
            fila[f"{m}_media"] = None
            fila[f"{m}_std"] = None
    filas_num.append(fila)

df_numerico = pd.DataFrame(filas_num)

# Ranking por F1-Score
print("\nRanking por F1-Score:")
ranking_f1 = df_numerico[
    ["Modelo", "F1_Score_media", "F1_Score_std"]
].sort_values("F1_Score_media", ascending=False)
print(ranking_f1.to_string(index=False))

# Ranking por AUC-ROC
print("\nRanking por AUC-ROC:")
ranking_auc = df_numerico[
    ["Modelo", "AUC_ROC_media", "AUC_ROC_std"]
].sort_values("AUC_ROC_media", ascending=False)
print(ranking_auc.to_string(index=False))

# Ranking por Tiempo de entrenamiento
print("\nRanking por Tiempo de entrenamiento (menor es mejor):")
ranking_tiempo = df_numerico[
    ["Modelo", "Tiempo_entrenamiento_s_media",
     "Tiempo_entrenamiento_s_std"]
].sort_values("Tiempo_entrenamiento_s_media", ascending=True)
print(ranking_tiempo.to_string(index=False))

# Ranking por velocidad de inferencia
print("\nRanking por Velocidad de inferencia (mayor es mejor):")
ranking_vel = df_numerico[
    ["Modelo", "Velocidad_inferencia_mps_media",
     "Velocidad_inferencia_mps_std"]
].sort_values("Velocidad_inferencia_mps_media", ascending=False)
print(ranking_vel.to_string(index=False))

TABLA NUMERICA — MEDIAS Y DESVIACIONES

Ranking por F1-Score:
          Modelo  F1_Score_media  F1_Score_std
LSTM Autoencoder          0.7872        0.0802
     Autoencoder          0.6441        0.1492
   One-Class SVM          0.2253        0.0079
Isolation Forest          0.1957        0.0348

Ranking por AUC-ROC:
          Modelo  AUC_ROC_media  AUC_ROC_std
LSTM Autoencoder         0.9633       0.0197
     Autoencoder         0.8743       0.0357
Isolation Forest         0.4788       0.0291
   One-Class SVM         0.2273       0.0003

Ranking por Tiempo de entrenamiento (menor es mejor):
          Modelo  Tiempo_entrenamiento_s_media  Tiempo_entrenamiento_s_std
   One-Class SVM                       10.0299                      0.4034
Isolation Forest                       18.1061                      2.6679
     Autoencoder                      385.1427                     83.8962
LSTM Autoencoder                     1083.5974                    193.0281

Ranking por Velocidad de 

In [ ]:
# ── CELDA 5: Respuestas a las RQ ─────────────────────────────
print("="*60)
print("RESPUESTAS A LAS PREGUNTAS DE INVESTIGACION")
print("="*60)

# Datos numéricos
datos = {}
for nombre, df in modelos.items():
    datos[nombre] = {
        m: df[m].mean() for m in metricas_vi + metricas_vd
        if m in df.columns
    }

print("\nRQ1: ¿Cómo influye el tiempo de entrenamiento")
print("     en la clasificación predictiva (F1-Score)?")
print("-"*50)
for nombre in modelos.keys():
    t = datos[nombre].get('Tiempo_entrenamiento_s', 0)
    f = datos[nombre].get('F1_Score', 0)
    print(f"  {nombre:20s}: {t:8.2f}s → F1={f:.4f}")

print("\nRQ2: ¿Qué relación existe entre velocidad de")
print("     inferencia y cuantificación de desviación?")
print("-"*50)
for nombre in modelos.keys():
    v = datos[nombre].get('Velocidad_inferencia_mps', 0)
    a = datos[nombre].get('AUC_ROC', 0)
    print(f"  {nombre:20s}: {v:10.0f} mps → AUC={a:.4f}")

print("\nRQ3: ¿Cómo influye el uso de memoria y CPU")
print("     en la confiabilidad del F1-Score?")
print("-"*50)
for nombre in modelos.keys():
    m = datos[nombre].get('Uso_memoria_MB', 0)
    f = datos[nombre].get('F1_Score', 0)
    print(f"  {nombre:20s}: {m:8.2f}MB → F1={f:.4f}")

print("\n" + "="*60)
print("CONCLUSION GENERAL")
print("="*60)
print("Modelo con mejor F1-Score  : LSTM Autoencoder (0.7872)")
print("Modelo con mejor AUC-ROC   : LSTM Autoencoder (0.9633)")
print("Modelo más rapido          : Isolation Forest (18.11s)")
print("Modelo más liviano         : One-Class SVM (~2 MB)")
print("Mejor trade-off costo/calidad: Autoencoder")
print("  → F1=0.6441 en 385s vs LSTM F1=0.7872 en 1083s")

RESPUESTAS A LAS PREGUNTAS DE INVESTIGACION

RQ1: ¿Cómo influye el tiempo de entrenamiento
     en la clasificación predictiva (F1-Score)?
--------------------------------------------------
  Isolation Forest    :    18.11s → F1=0.1957
  Autoencoder         :   385.14s → F1=0.6441
  One-Class SVM       :    10.03s → F1=0.2253
  LSTM Autoencoder    :  1083.60s → F1=0.7872

RQ2: ¿Qué relación existe entre velocidad de
     inferencia y cuantificación de desviación?
--------------------------------------------------
  Isolation Forest    :     212650 mps → AUC=0.4788
  Autoencoder         :      17086 mps → AUC=0.8743
  One-Class SVM       :      11258 mps → AUC=0.2273
  LSTM Autoencoder    :      16341 mps → AUC=0.9633

RQ3: ¿Cómo influye el uso de memoria y CPU
     en la confiabilidad del F1-Score?
--------------------------------------------------
  Isolation Forest    :    49.59MB → F1=0.1957
  Autoencoder         :   383.28MB → F1=0.6441
  One-Class SVM       :     1.93MB → F1=0.225

In [ ]:
# ── CELDA 6: Guardar tabla comparativa ───────────────────────
print("="*60)
print("GUARDANDO TABLA COMPARATIVA")
print("="*60)

# Tabla con formato media ± std
ruta_tabla = f"{ruta_comparacion}/tabla_comparativa_final.csv"
df_comparacion.to_csv(ruta_tabla)
print(f"✅ Tabla comparativa  : {ruta_tabla}")

# Tabla numérica
ruta_num = f"{ruta_comparacion}/tabla_numerica_final.csv"
df_numerico.to_csv(ruta_num, index=False)
print(f"✅ Tabla numerica     : {ruta_num}")

print("\n" + "="*60)
print("EXPERIMENTO COMPLETADO")
print("="*60)
print("Modelos evaluados : 4")
print("Repeticiones      : 5 por modelo")
print("Dataset           : CSE-CIC-IDS2018")
print("Training set      : 2,664,039 filas benignas")
print("Test set          : 4,967,041 filas mixtas")
print("Features          : 14 columnas")
print("GPU               : A100 (LSTM-AE)")
print(f"\n✅ Archivos guardados en: {ruta_comparacion}")

GUARDANDO TABLA COMPARATIVA
✅ Tabla comparativa  : /content/drive/MyDrive/IDS2018_resultados/comparacion/tabla_comparativa_final.csv
✅ Tabla numerica     : /content/drive/MyDrive/IDS2018_resultados/comparacion/tabla_numerica_final.csv

EXPERIMENTO COMPLETADO
Modelos evaluados : 4
Repeticiones      : 5 por modelo
Dataset           : CSE-CIC-IDS2018
Training set      : 2,664,039 filas benignas
Test set          : 4,967,041 filas mixtas
Features          : 14 columnas
GPU               : A100 (LSTM-AE)

✅ Archivos guardados en: /content/drive/MyDrive/IDS2018_resultados/comparacion
